# 03 - Feature Engineering

This notebook merges the two cleaned datasets and builds the feature matrix for anomaly detection.

**Steps:**
1. Aggregate `allarmi` to route level and left join with `viaggiatori` (0 rows lost)
2. Merge the two datasets via left join on route
3. Engineer rate features (alert rate, investigation rate, false positive rate) with sensible caps
4. Cleanup (drop metadata, sparse columns), encoding (binary, ordinal, one-hot with missing flags), imputation (median), scaling (StandardScaler)
5. Export unscaled and scaled feature matrices

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('project_anomaly_detection')

v = pd.read_csv(DATA_DIR / 'viaggiatori_clean.csv')
a = pd.read_csv(DATA_DIR / 'allarmi_clean.csv')

print('viaggiatori:', v.shape)
print('allarmi:    ', a.shape)

viaggiatori: (5095, 26)
allarmi:     (5080, 17)


## 1. Aggregate allarmi to route level

The two datasets have different granularity and don't align well on date — the same route can have different flight times recorded in each dataset.

**Strategy:** keep all 5095 rows from `viaggiatori` (no data loss), and enrich each row with route-level aggregated statistics from `allarmi`. This way we get per-check granularity with route-level context from the alarm system.

In [23]:
route_keys = ['AREOPORTO_PARTENZA', 'AREOPORTO_ARRIVO']

# pivot allarmi: each OCCORRENZE type becomes a column, aggregated per route
a_route = a.pivot_table(
    index=route_keys,
    columns='OCCORRENZE',
    values='TOT',
    aggfunc='sum'
).reset_index()

a_route.columns.name = None

# shorten column names
rename_map = {
    'Viaggiatori entrati nel Sistema':              'a_entrati',
    'Viaggiatori investigati':                      'a_investigati',
    'Viaggiatori con Allarmi':                      'a_allarmati',
    'Voli disponibili in ingresso al Sistema':      'a_voli_disponibili',
    'Voli con Allarmi':                             'a_voli_allarme',
    'Voli investigati (SDI/NSIS - INTERPOL - TSC)': 'a_voli_investigati',
    'Voli solo visualizzati, ma NON investigati':   'a_voli_solo_visual',
    'Allarmi generati da SDI/NSIS':                 'a_allarmi_sdi',
    'Allarmi generati da BCS':                      'a_allarmi_bcs',
    'Allarmi generati da INTERPOL':                 'a_allarmi_interpol',
    'Allarmi Chiusi':                               'a_chiusi',
    'Allarmi NON Chiusi':                           'a_non_chiusi',
    'Allarmi Rilevanti':                            'a_rilevanti',
    'Allarmi Chiusi con Azione (CC.xx)':            'a_chiusi_azione',
    'Nulla a procedere SDI':                        'a_nulla_sdi',
    'Nulla a procedere NSIS':                       'a_nulla_nsis',
    'Nulla a procedere INT':                        'a_nulla_int',
    'Errata segnalazione SDI':                      'a_errata_sdi',
    'Errata segnalazione NSIS':                     'a_errata_nsis',
    'Errata segnalazione BCS':                      'a_errata_bcs',
    'Notifica Atti/Provv':                          'a_notifica',
    'Respinto/a':                                   'a_respinto',
    'Allarmi generati':                             'a_allarmi_gen',
    'Mancato aggiornamento SDI':                    'a_manc_agg_sdi',
    'Mancato aggiornamento Schengen NSIS':          'a_manc_agg_nsis',
    'Inammissibilita Schengen - Art.24':            'a_inamm_schengen',
    'Altro':                                        'a_altro',
}

a_route.rename(columns=rename_map, inplace=True)

print('allarmi aggregated per route:', a_route.shape)
a_route.head()

allarmi aggregated per route: (368, 29)


,AREOPORTO_PARTENZA,AREOPORTO_ARRIVO,a_chiusi,a_chiusi_azione,a_non_chiusi,a_rilevanti,a_allarmi_gen,a_allarmi_bcs,a_allarmi_interpol,a_allarmi_sdi,...,a_nulla_nsis,a_nulla_sdi,a_respinto,a_allarmati,a_entrati,a_investigati,a_voli_allarme,a_voli_disponibili,a_voli_investigati,a_voli_solo_visual
0,ADB,MXP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,...,NaN,NaN,NaN,3.0,NaN,281.0,1.0,NaN,1.0,NaN
1,ADD,FCO,46.0,NaN,NaN,1.0,NaN,NaN,NaN,20.0,...,NaN,NaN,NaN,13.0,3047.0,0.0,NaN,4.0,NaN,7.0
2,ADD,MXP,52.0,NaN,NaN,NaN,NaN,NaN,NaN,197.0,...,NaN,NaN,NaN,30.0,663.0,873.0,2.0,1.0,3.0,1.0
3,AGA,BGY,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.0,...,NaN,NaN,NaN,NaN,478.0,NaN,NaN,NaN,NaN,1.0
4,ALA,MXP,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Merge: left join viaggiatori with route-level allarmi

Left join on route — every row of viaggiatori is kept, enriched with allarmi context where available.

In [24]:
df = v.merge(a_route, on=route_keys, how='left')

matched = df[df['a_entrati'].notna()].shape[0]
unmatched = df[df['a_entrati'].isna()].shape[0]

print(f'viaggiatori rows:    {len(v)}')
print(f'merged rows:         {len(df)}')
print(f'with allarmi data:   {matched} ({matched/len(df)*100:.1f}%)')
print(f'without allarmi:     {unmatched} ({unmatched/len(df)*100:.1f}%)')
print()
df.head()

viaggiatori rows:    5095
merged rows:         5095
with allarmi data:   4160 (81.6%)
without allarmi:     935 (18.4%)



,NAZIONALITA,AREOPORTO_ARRIVO,AREOPORTO_PARTENZA,ANNO_PARTENZA,MESE_PARTENZA,GIORNO_PARTENZA,DATA_PARTENZA,DESCR_AEREOPORTO_ARR,DESCR_AEREOPORTO_PART,CITTA_ARR,...,a_nulla_nsis,a_nulla_sdi,a_respinto,a_allarmati,a_entrati,a_investigati,a_voli_allarme,a_voli_disponibili,a_voli_investigati,a_voli_solo_visual
0,ALB,NAP,DUR,2024,2,13,2024-02-13 07:30:00,Napoli Capodichino,King Shaka International,Napoli,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,FCO,JFK,2024,1,22,2024-01-22 16:35:00,Fiumicino,John F Kennedy International,Roma,...,NaN,NaN,NaN,19.0,1090.0,1065.0,6.0,2.0,5.0,9.0
2,ALB,TSF,TIA,2024,2,4,2024-02-04 20:10:00,Treviso-Sant'Angelo,Rinas Mother Teresa,Treviso,...,NaN,32.0,NaN,160.0,1504.0,367.0,8.0,5.0,7.0,5.0
3,AFG,FCO,IST,2024,1,25,2024-01-25 13:05:00,Fiumicino,Havalimani,Roma,...,NaN,57.0,NaN,87.0,922.0,1407.0,7.0,4.0,4.0,11.0
4,ALB,BGY,MLE,2024,2,13,2024-02-13 00:00:00,Orio al Serio,Male International,Bergamo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Feature engineering

In [25]:
# rates from viaggiatori columns
df['alert_rate']         = df['ALLARMATI'] / df['ENTRATI']
df['investigation_rate'] = df['INVESTIGATI'] / df['ENTRATI']
df['alarm_per_invest']   = df['ALLARMATI'] / df['INVESTIGATI']

# rates from allarmi columns (route-level context)
df['false_positive_rate'] = (
    df[['a_nulla_sdi', 'a_nulla_nsis', 'a_nulla_int']].sum(axis=1) /
    df[['a_allarmi_sdi', 'a_allarmi_interpol']].sum(axis=1)
)

# replace inf with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# cap rates at sensible bounds:
# investigation_rate can't be >1 (can't investigate more people than entered)
# alert_rate and alarm_per_invest can be >1 (one person triggers multiple alarms)
#   but values like 34x are clearly data errors — cap at p99
# false_positive_rate can't be >1 (can't have more false positives than alarms)
df['investigation_rate'] = df['investigation_rate'].clip(upper=1.0)
df['alert_rate']         = df['alert_rate'].clip(upper=df['alert_rate'].quantile(0.99))
df['alarm_per_invest']   = df['alarm_per_invest'].clip(upper=df['alarm_per_invest'].quantile(0.99))
df['false_positive_rate'] = df['false_positive_rate'].clip(upper=1.0)

# closure_rate had near-zero variance (mean=0.993, std=0.045) — not useful, skip it

print(df[['alert_rate', 'investigation_rate', 'alarm_per_invest',
          'false_positive_rate']].describe().round(3))

       alert_rate  investigation_rate  alarm_per_invest  false_positive_rate
count    4961.000            4953.000          4705.000             3993.000
mean        0.249               0.945             0.253                0.381
std         0.476               0.225             0.467                0.309
min         0.000               0.000             0.000                0.000
25%         0.000               1.000             0.000                0.095
50%         0.087               1.000             0.102                0.359
75%         0.206               1.000             0.212                0.538
max         3.000               1.000             3.000                1.000


## 4. Cleanup, encoding, imputation and scaling

In [26]:
# drop text metadata columns — not useful for models
drop_cols = ['DESCR_AEREOPORTO_ARR', 'DESCR_AEREOPORTO_PART',
             'CITTA_ARR', 'CITTA_PARTENZA',
             'CODICE_PAESE_ARR', 'CODICE_PAESE_PART',
             'PAESE_ARR', 'PAESE_PART',
             'FLAG_TRANSITO', 'COMPAGNIA_AEREA', 'NUMERO_VOLO']

df.drop(columns=drop_cols, inplace=True)

# drop allarmi columns with >90% null — too sparse for models
sparse = [c for c in df.columns if c.startswith('a_') and df[c].isna().mean() > 0.90]
print('dropping sparse allarmi cols:', sparse)
df.drop(columns=sparse, inplace=True)

print('shape after cleanup:', df.shape)

dropping sparse allarmi cols: ['a_non_chiusi', 'a_allarmi_gen', 'a_allarmi_bcs', 'a_allarmi_interpol', 'a_altro', 'a_errata_bcs', 'a_errata_nsis', 'a_errata_sdi', 'a_inamm_schengen', 'a_manc_agg_sdi', 'a_manc_agg_nsis', 'a_nulla_int', 'a_nulla_nsis']
shape after cleanup: (5095, 33)


In [27]:
# temporal features from DATA_PARTENZA
df['DATA_PARTENZA'] = pd.to_datetime(df['DATA_PARTENZA'])
df['day_of_week'] = df['DATA_PARTENZA'].dt.dayofweek     # 0=Mon, 6=Sun
df['hour']        = df['DATA_PARTENZA'].dt.hour
df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)

# drop DATA_PARTENZA — we extracted what we need
df.drop(columns=['DATA_PARTENZA', 'ANNO_PARTENZA', 'GIORNO_PARTENZA'], inplace=True)

print('temporal features added')
print(df[['MESE_PARTENZA', 'day_of_week', 'hour', 'is_weekend']].head())

temporal features added
   MESE_PARTENZA  day_of_week  hour  is_weekend
0              2            1     7           0
1              1            0    16           0
2              2            6    20           1
3              1            3    13           0
4              2            1     0           0


In [28]:
# encode categorical columns

# GENERE: binary M=1, F=0 + missing flag
# without the flag, median imputation would assign all unknowns as F (biased)
df['genere_missing'] = df['GENERE'].isna().astype(int)
df['GENERE'] = df['GENERE'].map({'M': 1, 'F': 0})

# FASCIA_ETA: ordinal encoding + missing flag
df['fascia_missing'] = df['FASCIA_ETA'].isna().astype(int)
fascia_map = {'0-17': 0, '18-30': 1, '31-45': 2, '46-60': 3, '61+': 4}
df['FASCIA_ETA'] = df['FASCIA_ETA'].map(fascia_map)

# TIPO_DOCUMENTO: one-hot (NaN rows get all zeros = "unknown doc")
tipo_dummies = pd.get_dummies(df['TIPO_DOCUMENTO'], prefix='doc', dtype=int)
df = pd.concat([df, tipo_dummies], axis=1)
df.drop(columns=['TIPO_DOCUMENTO'], inplace=True)

# ESITO_CONTROLLO: one-hot (NaN rows get all zeros = "unknown outcome")
esito_dummies = pd.get_dummies(df['ESITO_CONTROLLO'], prefix='esito', dtype=int)
df = pd.concat([df, esito_dummies], axis=1)
df.drop(columns=['ESITO_CONTROLLO'], inplace=True)

# NAZIONALITA: top 5 + 'OTHER' then one-hot
top_naz = v['NAZIONALITA'].value_counts().head(5).index.tolist()
df['NAZIONALITA'] = df['NAZIONALITA'].where(df['NAZIONALITA'].isin(top_naz), 'OTHER')
naz_dummies = pd.get_dummies(df['NAZIONALITA'], prefix='naz', dtype=int)
df = pd.concat([df, naz_dummies], axis=1)
df.drop(columns=['NAZIONALITA'], inplace=True)

# drop airport codes — route info captured via allarmi features and zona
df.drop(columns=['AREOPORTO_PARTENZA', 'AREOPORTO_ARRIVO'], inplace=True)

print('shape after encoding:', df.shape)
print(df.dtypes.value_counts())

shape after encoding: (5095, 45)
float64    24
int64      19
int32       2
Name: count, dtype: int64


In [29]:
# impute NaN: median for numeric columns
from sklearn.impute import SimpleImputer

print('NaN before imputation:', df.isna().sum().sum())

imputer = SimpleImputer(strategy='median')
df_imputed = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)

print('NaN after imputation: ', df_imputed.isna().sum().sum())
print('shape:', df_imputed.shape)

NaN before imputation: 31030
NaN after imputation:  0
shape: (5095, 45)


In [30]:
# scale features using StandardScaler
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df_imputed), columns=df_imputed.columns)

print('scaled stats (should be ~0 mean, ~1 std):')
print(df_scaled.describe().loc[['mean', 'std']].round(2).T.head(10))

scaled stats (should be ~0 mean, ~1 std):
                 mean  std
MESE_PARTENZA    -0.0  1.0
ZONA              0.0  1.0
ENTRATI           0.0  1.0
INVESTIGATI       0.0  1.0
ALLARMATI         0.0  1.0
GENERE            0.0  1.0
FASCIA_ETA       -0.0  1.0
a_chiusi          0.0  1.0
a_chiusi_azione   0.0  1.0
a_rilevanti      -0.0  1.0


## 5. Export

In [31]:
# save both versions: unscaled (for interpretation) and scaled (for models)
df_imputed.to_csv(DATA_DIR / 'features.csv', index=False)
df_scaled.to_csv(DATA_DIR / 'features_scaled.csv', index=False)

print('features.csv (unscaled):', df_imputed.shape)
print('features_scaled.csv:    ', df_scaled.shape)
print()
print('columns:', list(df_scaled.columns))

features.csv (unscaled): (5095, 45)
features_scaled.csv:     (5095, 45)

columns: ['MESE_PARTENZA', 'ZONA', 'ENTRATI', 'INVESTIGATI', 'ALLARMATI', 'GENERE', 'FASCIA_ETA', 'a_chiusi', 'a_chiusi_azione', 'a_rilevanti', 'a_allarmi_sdi', 'a_notifica', 'a_nulla_sdi', 'a_respinto', 'a_allarmati', 'a_entrati', 'a_investigati', 'a_voli_allarme', 'a_voli_disponibili', 'a_voli_investigati', 'a_voli_solo_visual', 'alert_rate', 'investigation_rate', 'alarm_per_invest', 'false_positive_rate', 'day_of_week', 'hour', 'is_weekend', 'genere_missing', 'fascia_missing', "doc_Carta d'identità", 'doc_Passaporto', 'doc_Permesso di soggiorno', 'doc_Visto', 'esito_FERMATO', 'esito_IN ATTESA', 'esito_OK', 'esito_RESPINTO', 'esito_SEGNALATO', 'naz_AFG', 'naz_AGO', 'naz_ALB', 'naz_AND', 'naz_ARE', 'naz_OTHER']
